# Hardpoint breakaway tests recent history

In [ ]:
# Parameters
day_obs = 20250710

> **Note:**  
> This notebook was intended to track historical hardpoint test data.  
> Due to resource limitations, Times Square is not suitable for this.  
> Please download and adapt the notebook locally if needed.

In [ ]:
import asyncio
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.time import Time, TimeDelta
from datetime import datetime, timedelta

from lsst.summit.utils.efdUtils import makeEfdClient, getEfdData, getDayObsEndTime, getDayObsStartTime

In [ ]:
def stiffness(force, disp, name):
    """
    Compute stiffness and breakaway forces from force and displacement data.

    This function finds the maximum and minimum forces and corresponding displacements,
    then calculates the stiffness.
    It also returns the max and min forces.

    Args:
        force: Series of force values.
        disp: Series of displacement values, same index as `force`.
        name: Direction or mode indicator ('Pos' or 'Neg').

    Returns:
        tuple:
            breakaway_plus: Maximum force observed.
            breakaway_minus: Minimum force observed.
            stiffness: Computed stiffness (force/delta_disp).
                Returns NaN if displacement difference is zero or undefined.
    """
   
    forces = force.values
    disps = disp.values
    if name == 'Pos':
        maxi = np.argmax(force.values > 1000.0)
        mini = np.argmin(force.values < -1000.0)
    elif name == 'Neg':
        maxi = np.argmin(force.values > 1000.0)
        mini = np.argmax(force.values < -1000.0)
    else:
        raise  ValueError(f"Invalid name '{name}'. Expected 'Pos' or 'Neg'.")
        
    forceMax = forces[maxi] 
    forceMin = forces[mini]
    dispMax = disps[maxi] * 1.0E6
    dispMin = disps[mini] * 1.0E6

    delta_disp = dispMax - dispMin
    if np.isnan(delta_disp) or delta_disp == 0:
        print(f"Warning: Cannot compute stiffness — displacement difference between max and min is zero (dispMax={dispMax:.3f}, dispMin={dispMin:.3f})")
        stiffness = np.nan 
    else:
        stiffness = (forceMax - forceMin) / delta_disp
        
    breakaway_plus = np.max(forces)
    breakaway_minus = np.min(forces)
    return breakaway_plus, breakaway_minus, stiffness

def getStartsAndEnds(hardpoints, hp_index, previousState, startState, endState):
    """
    Identify start and end times of state transitions for a given hardpoint.

    Args:
        hardpoints: DataFrame with time-indexed test state data.
        hp_index: Index of the hardpoint to check.
        previousState: State that must precede the start state.
        startState: State indicating the start of the event.
        endState: State indicating the end of the event.

    Returns:
            starts: List of start times of valid transitions.
            ends: List of corresponding end times.

    Notes:
        - If the DataFrame is empty, a warning is printed and empty lists are returned.
        - If a start is found but no end transition occurs within the window,
          a message is printed and that event is skipped.
    """
    
    starts = []
    ends = []
    test_delay = 450
    
    if hardpoints.empty:
        print(f"[WARNING] No data available for hardpoint {hp_index}. Skipping.")
        return starts, ends
    
    for i in range(1, len(hardpoints)):
        try:
            if hardpoints.iloc[i][f'testState{hp_index}']==startState and \
            hardpoints.iloc[i - 1][f'testState{hp_index}']==previousState:
               
                this_start = hardpoints.index[i]
                test_time = this_start + pd.Timedelta(seconds=test_delay)
                k = 0
                
                # Avoid index out of bounds
                found = False
                while (i + k + 1) < len(hardpoints) and hardpoints.index[i + k] < test_time: 
                  if hardpoints.iloc[i + k][f'testState{hp_index}']==startState and \
                  hardpoints.iloc[i + k + 1][f'testState{hp_index}']==endState:
                     this_end = hardpoints.index[i + k + 1]
                     starts.append(Time(this_start, scale='utc'))
                     ends.append(Time(this_end, scale='utc'))
                     found = True
                     break
                  k += 1
                if not found:
                      print(f"[INFO] Hardpoint {hp_index}: start at {this_start}, but no end transition within the Time Delta: {test_delay} seconds.")
                  
        except KeyError as e:
            print(f"[WARNING] Column '{e.args[0]}' not found.")
            break
            
        except Exception as e:
            print(f"[ERROR] Unexpected error on hardpoint {hp_index} at index {i}: {e}")
            break                
            
    return starts, ends        

In [ ]:
# Create an EFD client instance
client = makeEfdClient()

# Define the start and end time
start_day_obs = day_obs
end_day_obs = day_obs
start_day_obs = 20250708
end_day_obs = 20250729

start_time = getDayObsStartTime(start_day_obs)
end_time = getDayObsEndTime(end_day_obs)
print(start_time, end_time)
hardpoints = await client.select_time_series(\
                    "lsst.sal.MTM1M3.logevent_hardpointTestStatus", \
                    ["*"], start_time, end_time)
print(len(hardpoints))

In [ ]:
%matplotlib inline

states = [['Pos',2,3,4], ['Neg',3,4,5]]

fig, axs = plt.subplots(1,2,figsize=(10,5))
fig.suptitle("MTM1M3 hardpoint trending", fontsize=18)
axs[0].set_title("Breakaway forces", fontsize=14)
axs[0].set_ylim(-4000, 4000)
axs[0].set_ylabel("Breakaway force (N)")
axs[1].set_title("Stiffness", fontsize=14)
axs[1].set_ylim(0, 100)
axs[1].set_ylabel("Stiffness (N/micron)")
min_time = 1.0E12
max_time = 0.0

data_found = False

for [name, previousState, startState, endState] in states:
    for hp_index in range(6):
        stiffs = []
        pos_breakaway = []
        neg_breakaway = []
        times = []
        starts, ends = getStartsAndEnds(hardpoints, hp_index, previousState, startState, endState)

        if len(starts) == 0 or len(ends) == 0:
            print(f"No data found for hardpoint {hp_index} in {name} mode.")
            continue
            
        for n in range(len(starts)):
            start = starts[n]
            #start = start - pd.Timedelta(seconds=600)
            end = ends[n]
            #end = end + pd.Timedelta(seconds=600)
            #print(hp_index, start, end)
            try:
                hardpointData = await client.select_time_series(\
                    "lsst.sal.MTM1M3.hardpointActuatorData", \
                    [f'displacement{hp_index}', f'measuredForce{hp_index}'], start, end)

                force = hardpointData[f'measuredForce{hp_index}']
                disp = hardpointData[f'displacement{hp_index}']
                forceMin, forceMax, stiff = stiffness(force, disp, name)
                pos_breakaway.append(forceMax)
                neg_breakaway.append(forceMin)
                stiffs.append(stiff)
                time = start.unix_tai
                times.append(time)
                if time < min_time:
                    min_time = time
                    min_time_list = start.isot.split('.')[0]
                if time > max_time:
                    max_time = time
                    max_time_list = start.isot.split('.')[0]
            except:
                continue

        if len(times) == 0:
            continue

        data_found = True
        
        if name == 'Pos':
            axs[0].plot(times, pos_breakaway, marker='x', label=f"{hp_index}")
            axs[0].plot(times, neg_breakaway, marker='x', label=f"{hp_index}")
        elif name == 'Neg':
            axs[0].plot(times, pos_breakaway, marker='x', label='')
            axs[0].plot(times, neg_breakaway, marker='x', label='')
        axs[1].plot(times, stiffs, marker='x', label=f"{name}_{hp_index}")
        axs[1].axhline(10, ls='--', color='black')
        axs[1].axhline(50, ls='--', color='black')
        
        axs[0].set_xticks([min_time, max_time],[min_time_list, max_time_list], rotation=10)
        axs[1].set_xticks([min_time, max_time],[min_time_list, max_time_list], rotation=10)

if data_found:
    axs[0].legend(loc='center left', ncol=3)
    axs[1].legend(loc='upper left', ncol=3)
    plt.show
    print(f"Analyzed stiffness and breakaway forces from {start_time} to {end_time}.")

else:
    print(f"No valid data found for any hardpoint from {start_time} to {end_time} — no plots were generated.")

In [ ]:
i = 3
hardpointData[f'measuredForce{hp_index}'].plot()

In [ ]:
hardpoints['testState3'].plot()